In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/puntos_reciclaje_geocodificado.csv")
df = df[df["latitude"].notna()].copy()
df["latitude"] = df["latitude"].astype(float)
df["longitude"] = df["longitude"].astype(float)

print(f"Puntos con coordenadas válidas: {len(df)}")
df[["N", "Distrito", "NombredelLugar", "latitude", "longitude"]]

Puntos con coordenadas válidas: 14


,N,Distrito,NombredelLugar,latitude,longitude
12,13,Miraflores,Tienda Races,-12.132897,-77.025575
23,24,Miraflores,Centro Comunal Santa Cruz,-12.176267,-76.970076
24,25,Miraflores,Casa del Adulto Mayor Tovar,-12.116842,-77.045707
29,30,Miraflores,Oficinas Municipales,-12.128359,-77.025132
36,37,Magdalena del Mar,Plaza Tupac Amaru,-12.091946,-77.073528
37,38,Magdalena del Mar,Mercado Magdalena,-12.090925,-77.073148
38,39,Magdalena del Mar,Boulevard,-12.104149,-77.060455
39,40,Magdalena del Mar,Supermercado Candy,-12.090758,-77.067113
40,41,Magdalena del Mar,Municipalidad de Magdalena del Mar,-12.091634,-77.067085
41,42,Magdalena del Mar,NaN,-12.092894,-77.061667


In [3]:
# Valores distritales — fuentes: INEI Censo 2017, APEIM 2022, SIGERSOL MINAM
valores_distrito = {
    "Miraflores": {
        "population_density": 8540,      # hab/km² — INEI Censo 2017
        "income_stratum": 5,             # escala 1-6 — APEIM
        "fuel_expenditure_sol": 65.0,    # soles/mes — ENAHO estimado
        "edu_level_head": 5.5,           # años promedio — INEI
        "household_size_avg": 2.8,       # personas/hogar — INEI
        "pct_nse_ab": 71.0,              # % NSE A+B — APEIM 2022
        "pct_nse_de": 4.0,               # % NSE D+E — APEIM 2022
        "gpc_kg_per_capita_day": 0.74,   # kg/día — SIGERSOL
        "pct_recyclable": 31.2,          # % — SIGERSOL
        "pct_plastic": 12.4,             # % — SIGERSOL
        "informal_recyclers_count": 14,  # estimado — padrón MINAM
        "slope_pct": 0.0,                # distrito plano costero
        "urbanized_area_pct": 95.0,      # % urbanizado
    },
    "Magdalena del Mar": {
        "population_density": 13200,
        "income_stratum": 4,
        "fuel_expenditure_sol": 55.0,
        "edu_level_head": 5.1,
        "household_size_avg": 3.1,
        "pct_nse_ab": 56.0,
        "pct_nse_de": 8.0,
        "gpc_kg_per_capita_day": 0.68,
        "pct_recyclable": 28.4,
        "pct_plastic": 11.1,
        "informal_recyclers_count": 8,
        "slope_pct": 0.0,
        "urbanized_area_pct": 95.0,
    },
}

In [4]:
feature_cols = list(valores_distrito["Miraflores"].keys())

for col in feature_cols:
    df[col] = df["Distrito"].map({k: v[col] for k, v in valores_distrito.items()})

df[feature_cols].describe()

,population_density,income_stratum,fuel_expenditure_sol,edu_level_head,household_size_avg,pct_nse_ab,pct_nse_de,gpc_kg_per_capita_day,pct_recyclable,pct_plastic,informal_recyclers_count,slope_pct,urbanized_area_pct
count,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.00000,14.000000,14.000000,14.0,14.0
mean,11868.571429,4.285714,57.857143,5.214286,3.014286,60.285714,6.857143,0.697143,29.20000,11.471429,9.714286,0.0,95.0
std,2184.641696,0.468807,4.688072,0.187523,0.140642,7.032108,1.875229,0.028128,1.31266,0.609449,2.812843,0.0,0.0
min,8540.000000,4.000000,55.000000,5.100000,2.800000,56.000000,4.000000,0.680000,28.40000,11.100000,8.000000,0.0,95.0
25%,9705.000000,4.000000,55.000000,5.100000,2.875000,56.000000,5.000000,0.680000,28.40000,11.100000,8.000000,0.0,95.0
50%,13200.000000,4.000000,55.000000,5.100000,3.100000,56.000000,8.000000,0.680000,28.40000,11.100000,8.000000,0.0,95.0
75%,13200.000000,4.750000,62.500000,5.400000,3.100000,67.250000,8.000000,0.725000,30.50000,12.075000,12.500000,0.0,95.0
max,13200.000000,5.000000,65.000000,5.500000,3.100000,71.000000,8.000000,0.740000,31.20000,12.400000,14.000000,0.0,95.0


In [5]:
# Asignar por distrito
for col, _ in list(valores_distrito["Miraflores"].items()):
    df[col] = df["Distrito"].map({k: v[col] for k, v in valores_distrito.items()})

In [6]:
from scipy.spatial.distance import cdist
import numpy as np

coords = df[["latitude", "longitude"]].values

# Distancia al punto más cercano (excluye sí mismo)
def dist_km(a, b):
    # Aproximación simple en grados → km (~111 km por grado en Lima)
    return np.sqrt(((a[0]-b[0])*111)**2 + ((a[1]-b[1])*88)**2)

dists = cdist(coords, coords, metric=lambda a,b: dist_km(a,b))
np.fill_diagonal(dists, np.inf)
df["dist_to_nearest_point_m"] = dists.min(axis=1) * 1000  # a metros

# Puntos existentes en radio 500m
df["existing_points_500m"] = (dists < 0.5).sum(axis=1)

# Índice de potencial de reciclaje
df["recycling_potential_index"] = (
    df["population_density"] * df["gpc_kg_per_capita_day"] * df["pct_recyclable"] / 100
)

In [7]:
# Estimados realistas para Miraflores/Magdalena — distritos urbanos densos
df["road_density"] = 0.035          # km de vía por km² — típico Lima urbano
df["dist_to_main_road_m"] = 150.0   # metros — distritos con avenidas densas
df["dist_to_market_m"] = 400.0      # metros — hay mercados cada ~400m
df["has_park_300m"] = 1             # Miraflores tiene parques por todos lados

In [11]:
# Estos 14 son puntos REALES de reciclaje → todos positivos
df["is_suitable"] = 1

# Columnas finales
FEATURES = [
    "population_density", "income_stratum", "fuel_expenditure_sol",
    "edu_level_head", "household_size_avg", "pct_nse_ab", "pct_nse_de",
    "gpc_kg_per_capita_day", "pct_recyclable", "pct_plastic",
    "informal_recyclers_count", "road_density", "dist_to_main_road_m",
    "dist_to_market_m", "dist_to_nearest_point_m", "existing_points_500m",
    "has_park_300m", "slope_pct", "urbanized_area_pct",
    "recycling_potential_index"
]

df[FEATURES + ["is_suitable", "latitude", "longitude"]].to_csv(
    "../data/real/puntos_positivos_v1.csv", index=False
)
print(f"Guardado: {len(df)} puntos positivos")

Guardado: 14 puntos positivos


In [12]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import box

# Rejilla de celdas sobre los 2 distritos
miraflores = ox.geocode_to_gdf("Miraflores, Lima, Peru")
magdalena = ox.geocode_to_gdf("Magdalena del Mar, Lima, Peru")
area = pd.concat([miraflores, magdalena]).to_crs(epsg=32718)

xmin, ymin, xmax, ymax = area.total_bounds
celdas = [box(x, y, x+500, y+500)
          for x in np.arange(xmin, xmax, 500)
          for y in np.arange(ymin, ymax, 500)]

grid = gpd.GeoDataFrame(geometry=celdas, crs="EPSG:32718")
grid = gpd.clip(grid, area).to_crs(epsg=4326)
grid["centroid_lat"] = grid.geometry.centroid.y
grid["centroid_lon"] = grid.geometry.centroid.x

# Marcar como negativo toda celda sin punto real a ≤300m
puntos_positivos = df[["latitude", "longitude"]].values
def tiene_punto(lat, lon):
    for p in puntos_positivos:
        d = dist_km([lat, lon], p)
        if d <= 0.3:
            return True
    return False

grid["is_suitable"] = grid.apply(
    lambda r: 1 if tiene_punto(r["centroid_lat"], r["centroid_lon"]) else 0, axis=1
)
# Asigna features distritales igual que arriba...
# Toma muestra balanceada: todos los positivos + 2-3x negativos
neg = grid[grid["is_suitable"] == 0].sample(min(40, len(grid)))
pos = grid[grid["is_suitable"] == 1]
dataset_final = pd.concat([pos, neg])
print(dataset_final["is_suitable"].value_counts())

is_suitable
0    40
1    18
Name: count, dtype: int64


C:\Users\nikole.garcia\AppData\Local\Temp\ipykernel_25336\3812956270.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid["centroid_lat"] = grid.geometry.centroid.y
C:\Users\nikole.garcia\AppData\Local\Temp\ipykernel_25336\3812956270.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid["centroid_lon"] = grid.geometry.centroid.x


In [13]:
import pandas as pd
import numpy as np

# ── Asignar features distritales a las celdas del grid ───────────────────────
# El grid tiene centroid_lat / centroid_lon — determinar distrito por bounding box
# Magdalena del Mar: lat < -12.085 y lon < -77.055  (aprox)
def asignar_distrito(row):
    if row["centroid_lon"] < -77.055:
        return "Magdalena del Mar"
    return "Miraflores"

grid["Distrito"] = grid.apply(asignar_distrito, axis=1)

for col in feature_cols:
    grid[col] = grid["Distrito"].map({k: v[col] for k, v in valores_distrito.items()})

# Features espaciales del grid
coords_grid = grid[["centroid_lat", "centroid_lon"]].values
dists_grid = cdist(coords_grid, coords, metric=lambda a, b: dist_km(a, b))
grid["dist_to_nearest_point_m"] = dists_grid.min(axis=1) * 1000
grid["existing_points_500m"]    = (dists_grid < 0.5).sum(axis=1)
grid["recycling_potential_index"] = (
    grid["population_density"] * grid["gpc_kg_per_capita_day"] * grid["pct_recyclable"] / 100
)
grid["road_density"]         = 0.035
grid["dist_to_main_road_m"]  = 150.0
grid["dist_to_market_m"]     = 400.0
grid["has_park_300m"]        = 1

# ── Combinar positivos reales + negativos del grid ───────────────────────────
pos_df = df.copy()
pos_df["centroid_lat"] = pos_df["latitude"]
pos_df["centroid_lon"] = pos_df["longitude"]

neg = grid[grid["is_suitable"] == 0].sample(min(40, (grid["is_suitable"] == 0).sum()), random_state=42)
pos = grid[grid["is_suitable"] == 1]  # 18 positivos del grid (celdas con punto real)

# Usar solo los positivos del CSV real (más confiables que los del grid)
COLS = feature_cols + ["dist_to_nearest_point_m", "existing_points_500m",
                       "recycling_potential_index", "road_density",
                       "dist_to_main_road_m", "dist_to_market_m",
                       "has_park_300m", "is_suitable"]

combined = pd.concat([
    pos_df[COLS + ["latitude", "longitude"]].assign(centroid_lat=pos_df["latitude"], centroid_lon=pos_df["longitude"]),
    neg[COLS + ["centroid_lat", "centroid_lon"]].rename(columns={"centroid_lat": "latitude", "centroid_lon": "longitude"})
]).reset_index(drop=True)

# ── Renombrar para que coincidan con config.py ────────────────────────────────
rename_map = {
    "is_suitable":              "is_optimal",
    "pct_nse_ab":               "nse_ab_pct",
    "pct_nse_de":               "nse_de_pct",
    "dist_to_nearest_point_m":  "dist_nearest_recycling_m",
    "existing_points_500m":     "recycling_density_1km",
    "dist_to_main_road_m":      "dist_nearest_road_m",
    "gpc_kg_per_capita_day":    "waste_per_capita_kg",
}
combined = combined.rename(columns=rename_map)
combined["urbanization_rate"] = combined["urbanized_area_pct"] / 100

# Columnas requeridas por config.py que no tenemos → agregar con estimados
district_extras = {
    "Miraflores":        {"num_households": 28000, "nse_c_pct": 25.0, "walkability_score": 0.80,
                          "poi_commercial_500m": 8, "poi_educational_500m": 3, "land_use_encoded": 1, "area_m2": 600},
    "Magdalena del Mar": {"num_households": 22000, "nse_c_pct": 36.0, "walkability_score": 0.75,
                          "poi_commercial_500m": 6, "poi_educational_500m": 2, "land_use_encoded": 1, "area_m2": 500},
}
# Recuperar Distrito desde pct_nse_ab original (ya renombrado a nse_ab_pct)
combined["Distrito"] = combined["nse_ab_pct"].map({71.0: "Miraflores", 56.0: "Magdalena del Mar"})
for extra_col in ["num_households", "nse_c_pct", "walkability_score",
                  "poi_commercial_500m", "poi_educational_500m", "land_use_encoded", "area_m2"]:
    combined[extra_col] = combined["Distrito"].map({k: v[extra_col] for k, v in district_extras.items()})

# district_id para spatial CV
combined["district_id"] = combined["Distrito"].map({"Miraflores": 1, "Magdalena del Mar": 2})

# ── Guardar ───────────────────────────────────────────────────────────────────
out_path = "../data/real/dataset_entrenamiento_v1.csv"
combined.to_csv(out_path, index=False)
print(f"Guardado: {len(combined)} filas en {out_path}")
print(combined["is_optimal"].value_counts())
print(f"Columnas: {list(combined.columns)}")

Guardado: 54 filas en ../data/real/dataset_entrenamiento_v1.csv
is_optimal
0    40
1    14
Name: count, dtype: int64
Columnas: ['population_density', 'income_stratum', 'fuel_expenditure_sol', 'edu_level_head', 'household_size_avg', 'nse_ab_pct', 'nse_de_pct', 'waste_per_capita_kg', 'pct_recyclable', 'pct_plastic', 'informal_recyclers_count', 'slope_pct', 'urbanized_area_pct', 'dist_nearest_recycling_m', 'recycling_density_1km', 'recycling_potential_index', 'road_density', 'dist_nearest_road_m', 'dist_to_market_m', 'has_park_300m', 'is_optimal', 'latitude', 'longitude', 'centroid_lat', 'centroid_lon', 'urbanization_rate', 'Distrito', 'num_households', 'nse_c_pct', 'walkability_score', 'poi_commercial_500m', 'poi_educational_500m', 'land_use_encoded', 'area_m2', 'district_id']
